In [69]:
from typing import TypedDict
from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END

from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.messages import HumanMessage

load_dotenv()

True

In [1]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv()


class HuggingFaceLLM:

    def __init__(
        self,
        model_name="Qwen/Qwen2.5-7B-Instruct",
        api_key=None,
    ):

        self.model_name = model_name

        self.client = InferenceClient(
            api_key=api_key or os.getenv("HF_TOKEN")
        )

    def invoke(self, prompt: str) -> str:

        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        return response.choices[0].message.content
    
llm = HuggingFaceLLM()

e:\Pradhumn- DS\Agentic AI using Langgraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [60]:
class State(TypedDict):
    topic: str
    draft: str
    improved: str
    summary: str

In [61]:
#node 1
def generate(state: State):

    prompt = f"""Explain the following topic in simple English.Topic: {state['topic']}"""
    response = llm.invoke(prompt)

    return {"draft": response}

In [62]:
#node 2
def improve(state: State):
    
    prompt = f"""Improve the following explanation.{state['draft']}"""

    response = llm.invoke(prompt)

    return {"improved": response}


In [63]:
#node 3
def summarize(state: State):

    prompt = f"""Summarize the following into 5 bullet points.{state['improved']}"""
    response = llm.invoke(prompt)

    return {"summary": response}

In [64]:
builder = StateGraph(State)

builder.add_node("generate", generate)
builder.add_node("improve", improve)
builder.add_node("summary", summarize)

builder.add_edge(START, "generate")
builder.add_edge("generate", "improve")
builder.add_edge("improve", "summary")
builder.add_edge("summary", END)

In [65]:
#complie graph
graph = builder.compile()

In [75]:
result = graph.invoke({"topic": "Machine Learning"})

In [76]:
print("\nDraft:\n")
print(result["draft"])



Draft:

Sure! Machine learning is like teaching a computer to learn on its own, without being explicitly programmed. Imagine you have a big box of toys, and you want the computer to sort them into different groups. Instead of writing a long list of rules for the computer to follow (like "put all red toys in one pile and all blue toys in another"), you show the computer lots of examples of toys and tell it which group each toy belongs to. Over time, the computer starts to understand the differences between the toys and can sort new toys into the right groups all by itself. That's a simple way to think about machine learning!


In [ ]:
print("\nImproved:\n")
print(result["improved"])


Improved:

Certainly! Here's an improved version of the explanation:

Machine learning is a powerful technique that enables computers to learn and improve from experience or data, without being explicitly programmed. Imagine you have a large collection of toys, and your goal is to sort them into different groups. Instead of providing the computer with a detailed set of rules (such as "put all red toys in one pile and all blue toys in another"), you can show the computer many examples of toys and their corresponding groups. By analyzing these examples, the computer begins to understand the characteristics that define each group. Over time, it can sort new toys into the correct groups on its own, without needing further instructions. This process of learning from examples and making predictions or decisions based on that learning is the essence of machine learning.


In [78]:
print("\nSummary:\n")
print(result["summary"])


Summary:

- Machine learning allows computers to learn and improve from experience or data without explicit programming.
- It involves showing the computer examples of data and their corresponding outcomes to help it understand patterns.
- The computer then uses this understanding to make predictions or decisions on new, unseen data.
- This process enables the computer to sort and categorize items (like toys) into predefined groups autonomously.
- Machine learning is based on the idea of learning from examples rather than following predefined rules.
